# 04. Ortholog Inference via BLAST, MSA, and Phylogenetic Distance

Searches each candidate protein against the Arabidopsis proteome with BLAST, builds a multiple sequence alignment and phylogenetic tree per candidate, and determines the closest Arabidopsis ortholog by tree distance.

**Input**: `dataset/blast/tomato_candidates.fasta`, `dataset/blast/GCF_000001735.4_TAIR10.1_protein.faa`

**Output**: `dataset/processed/ortholog_summary.csv`

In [25]:
import os
import pandas as pd
from Bio import SeqIO
from Bio import Phylo
import matplotlib.pyplot as plt
import glob

## BLAST result

blastp was run externally against a local Arabidopsis TAIR10.1 protein database (see `scripts/03_run_blastp.sh`), since Arabidopsis is well-annotated and serves as a reference for inferring tomato gene function.

In [10]:
cols = ["query", "subject", "identity", "length", "mismatch", "gapopen",
        "qstart", "qend", "sstart", "send", "evalue", "bitscore"]
results = pd.read_csv("../dataset/blast/blast_results.tsv", sep="\t", header=None, names=cols)
results

,query,subject,identity,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
0,Solyc05g006640.4,NP_176191,76.377,1198,277,5,8,1203,20,1213,0.0,1958.0
1,Solyc05g006640.4,NP_177038,47.396,1133,551,17,17,1117,34,1153,0.0,1025.0
2,Solyc05g006640.4,NP_001320717,47.396,1133,551,17,17,1117,34,1153,0.0,1025.0
3,Solyc05g006640.4,NP_189189,47.294,1127,558,17,20,1120,35,1151,0.0,1015.0
4,Solyc05g006640.4,NP_001322420,45.863,1136,579,14,24,1134,42,1166,0.0,1012.0
...,...,...,...,...,...,...,...,...,...,...,...,...
71,Solyc05g051220.3,NP_186934,71.658,808,226,2,18,823,22,828,0.0,1218.0
72,Solyc05g051220.3,NP_198566,68.234,787,242,5,46,824,31,817,0.0,1133.0
73,Solyc05g051220.3,NP_001332118,66.378,809,242,6,46,824,31,839,0.0,1120.0
74,Solyc05g051220.3,NP_001325630,73.315,727,191,2,18,742,22,747,0.0,1118.0


## Grouping top BLAST hits per candidate

Each candidate's top 5 Arabidopsis hits form its own comparison group; this does not correspond to a shared protein family across candidates, since every candidate turned out to have distinct top hits.

In [14]:
# Keep the top 5 Arabidopsis hits per candidate, ranked by evalue then bitscore, 
# to build a manageable per-candidate alignment group
top_n = (
    results.sort_values(["evalue", "bitscore"], ascending=[True, False])
    .groupby("query")
    .head(5) # Top 5 hit / query
)

family_groups = top_n.groupby("query")["subject"].apply(list).to_dict()
family_groups

{'Solyc01g006150.3': ['NP_191365',
  'NP_001319789',
  'NP_001320580',
  'NP_001320579',
  'NP_178070'],
 'Solyc01g105450.4': ['NP_173226',
  'NP_188746',
  'NP_175561',
  'NP_175557',
  'NP_850111'],
 'Solyc01g106900.3': ['NP_188598',
  'NP_563988',
  'NP_001185007',
  'NP_001077545',
  'NP_565247'],
 'Solyc02g082450.3': ['NP_973479',
  'NP_849964',
  'NP_565417',
  'NP_001031363',
  'NP_201399'],
 'Solyc02g092450.3': ['NP_195479',
  'NP_179879',
  'NP_849716',
  'NP_001321522',
  'NP_001321521'],
 'Solyc05g006020.3': ['NP_176211',
  'NP_851251',
  'NP_201094',
  'NP_001032127',
  'NP_172524'],
 'Solyc05g006640.4': ['NP_176191',
  'NP_177038',
  'NP_001320717',
  'NP_189189',
  'NP_001322420'],
 'Solyc05g009020.4': ['NP_564233',
  'NP_001321077',
  'NP_566102',
  'NP_569032',
  'NP_001328407'],
 'Solyc05g051220.3': ['NP_186934',
  'NP_198566',
  'NP_001332118',
  'NP_001325630',
  'NP_001332117'],
 'Solyc07g063520.3': ['NP_193888',
  'NP_001329391',
  'NP_563884',
  'NP_565152',
  'NP

In [19]:
arabidopsis_fasta = SeqIO.to_dict(SeqIO.parse("../dataset/blast/GCF_000001735.4_TAIR10.1_protein.faa", "fasta"))

# NCBI RefSeq IDs carry a version suffix not present in the BLAST subject IDs, so it is stripped before mapping
ara_id2seq = {k.split(".")[0]: str(v.seq) for k, v in arabidopsis_fasta.items()}

candidates = pd.read_csv("../dataset/processed/final_candidates_with_seq.csv")
tomato_id2seq = dict(zip(candidates["names"], candidates["seq"]))

os.makedirs("../dataset/msa/groups", exist_ok=True)

for query, group_df in top_n.groupby("query"):
    out_path = f"../dataset/msa/groups/{query}.fasta"
    with open(out_path, "w") as f:
        f.write(f">{query}\n{tomato_id2seq[query]}\n")
        for subj in group_df["subject"]:
            f.write(f">{subj}\n{ara_id2seq[subj]}\n")

## Alignment and tree construction

Each group fasta was aligned with MAFFT and used to build a phylogenetic tree with FastTree (see `scripts/04_run_mafft.sh`, `scripts/05_run_fasttree.sh`).

In [21]:
tree = Phylo.read("../dataset/trees/Solyc05g006640.4.nwk", "newick")
Phylo.draw_ascii(tree)

     , NP_177038
  ___|
 |   | NP_001320717
 |
_|______ NP_001322420
 |
 |     _______ NP_189189
 |____|
      |                                              _______ Solyc05g006640.4
      |_____________________________________________|
                                                    |________ NP_176191



In [31]:
ara_id2label = {
    record.id.split(".")[0]: record.description.split(" ", 1)[1] if " " in record.description else record.id
    for record in SeqIO.parse("../dataset/blast/GCF_000001735.4_TAIR10.1_protein.faa", "fasta")
}

# Lookup used to relabel tree leaves with functional descriptions for readability
id2label = dict(zip(candidates["names"], candidates["description"]))
id2label.update(ara_id2label)

tree = Phylo.read("../dataset/trees/Solyc05g006640.4.nwk", "newick")
for leaf in tree.get_terminals():
    if leaf.name in id2label:
        leaf.name = f"{leaf.name} ({id2label[leaf.name]})"

Phylo.draw_ascii(tree)

   , NP_177038 (ATPase E1-E2 type family p...
  _|
 | | NP_001320717 (ATPase E1-E2 type famil...
 |
_|___ NP_001322420 (ATPase E1-E2 type famil...
 |
 |   ____ NP_189189 (ATPase E1-E2 type family p...
 |__|
    |                           ____ Solyc05g006640.4 (Phospholipid-transp...
    |__________________________|
                               |____ NP_176191 (aminophospholipid ATPase 3...



In [32]:
all_rows = []

for path in glob.glob("../dataset/trees/*.nwk"):
    query_name = path.split("/")[-1].replace(".nwk", "")
    tree = Phylo.read(path, "newick")

    for leaf in tree.get_terminals():
        if leaf.name == query_name:
            continue
        dist = tree.distance(query_name, leaf.name)
        all_rows.append({
            "tomato_candidate": query_name,
            "tomato_description": id2label.get(query_name, ""),
            "arabidopsis_id": leaf.name,
            "arabidopsis_description": id2label.get(leaf.name, ""),
            "tree_distance": dist,
        })

summary_all = pd.DataFrame(all_rows).sort_values(["tomato_candidate", "tree_distance"])
summary_all

,tomato_candidate,tomato_description,arabidopsis_id,arabidopsis_description,tree_distance
70,Solyc01g006150.3,Cation efflux protein (AHRD V3.3 *** A0A200PY7...,NP_191365,Cation efflux family protein [Arabidopsis thal...,0.424466
71,Solyc01g006150.3,Cation efflux protein (AHRD V3.3 *** A0A200PY7...,NP_001319789,Cation efflux family protein [Arabidopsis thal...,0.424466
74,Solyc01g006150.3,Cation efflux protein (AHRD V3.3 *** A0A200PY7...,NP_178070,Cation efflux family protein [Arabidopsis thal...,1.012938
72,Solyc01g006150.3,Cation efflux protein (AHRD V3.3 *** A0A200PY7...,NP_001320580,Cation efflux family protein [Arabidopsis thal...,1.020873
73,Solyc01g006150.3,Cation efflux protein (AHRD V3.3 *** A0A200PY7...,NP_001320579,Cation efflux family protein [Arabidopsis thal...,1.020873
...,...,...,...,...,...
65,Solyc11g062020.2,Zinc transporter ZTP29 (AHRD V3.3 *** A0A2G3B9...,NP_566669,ZIP metal ion transporter family [Arabidopsis ...,0.160060
66,Solyc11g062020.2,Zinc transporter ZTP29 (AHRD V3.3 *** A0A2G3B9...,NP_001326769,ZIP metal ion transporter family [Arabidopsis ...,0.160060
67,Solyc11g062020.2,Zinc transporter ZTP29 (AHRD V3.3 *** A0A2G3B9...,NP_001326770,ZIP metal ion transporter family [Arabidopsis ...,0.160060
68,Solyc11g062020.2,Zinc transporter ZTP29 (AHRD V3.3 *** A0A2G3B9...,NP_187477,ZIP metal ion transporter family [Arabidopsis ...,1.501663


The Arabidopsis description in `best_match` (saved as `ortholog_summary.csv`) is the single closest ortholog per candidate, determined by tree distance rather than raw BLAST score alone. This value is later used directly as the sequence-based functional prediction in notebook 05's cross-validation with Foldseek.

In [33]:
blast_lookup = results.set_index(["query", "subject"])[["identity", "evalue", "length"]]
summary_all["arabidopsis_id_clean"] = summary_all["arabidopsis_id"].str.split(" ").str[0]

merged = summary_all.merge(
    blast_lookup,
    left_on=["tomato_candidate", "arabidopsis_id_clean"],
    right_index=True,
    how="left",
)

best_match = merged.sort_values(["tomato_candidate", "tree_distance"]).groupby("tomato_candidate").first().reset_index()

best_match[["tomato_candidate", "tomato_description", "arabidopsis_id_clean",
            "arabidopsis_description", "tree_distance", "identity", "evalue"]]

,tomato_candidate,tomato_description,arabidopsis_id_clean,arabidopsis_description,tree_distance,identity,evalue
0,Solyc01g006150.3,Cation efflux protein (AHRD V3.3 *** A0A200PY7...,NP_191365,Cation efflux family protein [Arabidopsis thal...,0.424466,69.171,0.000000e+00
1,Solyc01g105450.4,ABC transporter G family member 11 (AHRD V3.3 ...,NP_173226,white-brown complex-like protein [Arabidopsis ...,0.840516,52.188,0.000000e+00
2,Solyc01g106900.3,Magnesium transporter MRS2-F (AHRD V3.3 *** A0...,NP_188598,magnesium transporter 4 [Arabidopsis thaliana],0.489231,64.659,0.000000e+00
3,Solyc02g082450.3,Auxin efflux carrier (AHRD V3.3 *** A0A200PYL2...,NP_201399,Auxin efflux carrier family protein [Arabidops...,0.554919,61.235,0.000000e+00
4,Solyc02g092450.3,Calcium-transporting ATPase (AHRD V3.3 *** A0A...,NP_195479,calcium ATPase 2 [Arabidopsis thaliana],0.212021,82.220,0.000000e+00
5,Solyc05g006020.3,Nucleobase-ascorbate transporter 8 (AHRD V3.3 ...,NP_176211,nucleobase-ascorbate transporter 7 [Arabidopsi...,0.244818,80.835,0.000000e+00
6,Solyc05g006640.4,Phospholipid-transporting ATPase (AHRD V3.3 **...,NP_176191,aminophospholipid ATPase 3 [Arabidopsis thaliana],0.301098,76.377,0.000000e+00
7,Solyc05g009020.4,Mitochondrial carrier protein (AHRD V3.3 *** A...,NP_566102,NAD+ transporter 1 [Arabidopsis thaliana],0.516293,64.211,5.160000e-127
8,Solyc05g051220.3,Potassium channel GORK (AHRD V3.3 *** A0A1U8GY...,NP_186934,STELAR K+ outward rectifier [Arabidopsis thali...,0.406637,71.658,0.000000e+00
9,Solyc07g063520.3,Organic solute transporter ostalpha protein (D...,NP_193888,organic solute transporter ostalpha protein (D...,0.384270,69.257,2.010000e-152


In [34]:
best_match.to_csv("../dataset/processed/ortholog_summary.csv", index=False)

In [35]:
candidates["seq_len"] = candidates["seq"].str.len()
candidates[["names", "seq_len"]].sort_values("seq_len", ascending=False)

,names,seq_len
10,Solyc09g075020.3,1514
0,Solyc05g006640.4,1205
2,Solyc02g092450.3,1018
14,Solyc05g051220.3,825
12,Solyc09g065560.3,659
13,Solyc01g105450.4,631
3,Solyc05g006020.3,534
8,Solyc01g106900.3,499
1,Solyc02g082450.3,406
6,Solyc01g006150.3,406
